# 13 — Cross-evaluation: train without 1128(b)(4), test with it

**Question.** Notebook 11 removed license-revocation (b4) positives from *both* training and testing (AUC 0.763 → 0.804). Is the gain a better model, or just an easier test? Here the two are separated.

**Design** (combined B+D, B+D features, XGBoost, paper settings):
- Folds are assigned **once per run** from the paper label (`fraud`), by NPI, stratified by ever-fraud, so b4 providers are spread across test folds. Both models below use **the same folds**, so differences are paired.
- **Model `train_full`:** trained on all training rows with the paper label.
- **Model `train_nob4`:** trained on the same rows **minus the b4 fraud rows** (the rows Notebook 10 marked `fraud_nob4 = -1`).
- Every model is scored on two test sets built from the same test fold:
  - `test_full` — all rows, paper label (b4 providers count as fraud)
  - `test_nob4` — b4 fraud rows removed (Notebook 11's test)
- Scored overall and per provider group (both / B only / D only).
- **By exclusion type:** on `test_full`, AUC of each type's positives against all negatives. A type near 0.5 is not detectable from billing.

| Train \ Test | `test_full` | `test_nob4` |
|---|---|---|
| `train_full` | ≈ Notebook 11 `fraud` (0.763) | is the gain just an easier test? |
| `train_nob4` | **is it a better model?** | ≈ Notebook 11 `fraud_nob4` (0.804) |

Numbers in the two diagonal cells differ slightly from Notebook 11 because folds here are assigned from `fraud` for both models.

In [1]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

SMOKE  = os.environ.get("SMOKE", "1") == "1"
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
BD       = DATA_DIR / "processed/combined/bd_labeled_2019_2023.parquet"
LEIE_CUM = DATA_DIR / "processed/part_b/labeled/leie_cumulative_2019_2023.parquet"
RES_DIR  = DATA_DIR / "results/combined"; RES_DIR.mkdir(parents=True, exist_ok=True)
TAG      = "nb13_crosseval" + ("_smoke" if SMOKE else "")
RES_CSV  = RES_DIR / f"{TAG}_fold_results.csv"

N_RUNS, N_FOLDS = (1, 5) if SMOKE else (6, 5)
FOLDS_TO_RUN    = [0] if SMOKE else list(range(N_FOLDS))
TRAIN_MODELS    = ["train_full", "train_nob4"]
print("SMOKE:", SMOKE, "| N_JOBS:", N_JOBS, "| results →", RES_CSV.name)

def make_model(seed):
    return XGBClassifier(max_depth=4, n_jobs=N_JOBS, random_state=seed, tree_method="hist", eval_metric="logloss")

SMOKE: False | N_JOBS: 16 | results → nb13_crosseval_fold_results.csv


## Step 1 — Data, labels and exclusion types

In [2]:
df = pd.read_parquet(BD)
print(df.shape)

cum = pd.read_parquet(LEIE_CUM, columns=["NPI", "EXCLTYPE"])
assert cum.NPI.is_unique
excl = df["npi"].map(cum.set_index("NPI").EXCLTYPE)          # exclusion type of the provider (NaN = never excluded)

y_all   = df["fraud"].to_numpy(np.int8)                       # paper label, used for every test
is_b4   = (df["fraud_nob4"] == -1).to_numpy()                 # b4 fraud rows (dropped by the no-b4 label)
assert (is_b4 == ((y_all == 1) & (excl == "1128b4").to_numpy())).all(), "b4 flag does not match LEIE exclusion type"
pos_type = np.where(y_all == 1, excl.fillna("none").to_numpy(), "neg")
grp = df["group"].to_numpy()

print("fraud rows:", int(y_all.sum()), "| of which b4:", int(is_b4.sum()))
print(pd.Series(pos_type[y_all == 1]).value_counts())

(7425062, 177)


fraud rows: 3958 | of which b4: 1799
1128b4    1799
1128a1    1522
1128a3     342
1128a2     172
1128b7     123
Name: count, dtype: int64


In [3]:
# B+D feature matrix, built exactly as in Notebook 11.
B_FEATS = [c for c in df.columns if c.startswith("b_") and c not in ("b_type", "b_ent_cd")]
D_FEATS = [c for c in df.columns if c.startswith("d_") and c != "d_type"]
FLAGS   = ["has_partb", "has_partd"]
b_pt = pd.get_dummies(df["b_type"].where(df.has_partb == 1), prefix="bpt", dtype=np.float32)
d_pt = pd.get_dummies(df["d_type"].where(df.has_partd == 1), prefix="dpt", dtype=np.float32)
X_all = np.hstack([df[B_FEATS + D_FEATS + FLAGS].to_numpy(np.float32),
                   b_pt.to_numpy(np.float32), d_pt.to_numpy(np.float32)])
del b_pt, d_pt
print("X_all:", X_all.shape, f"{X_all.nbytes/1e9:.1f} GB")

npi = df["npi"].to_numpy()
npi_ever = df.groupby("npi")["fraud"].max()
del df

def assign_folds(seed):
    rng = np.random.default_rng(seed)
    parts = []
    for cls in (0, 1):
        ids = rng.permutation(npi_ever.index[npi_ever == cls].to_numpy())
        parts.append(pd.Series(np.arange(len(ids)) % N_FOLDS, index=ids))
    return pd.Series(npi).map(pd.concat(parts)).to_numpy()

f0 = assign_folds(0)
chk = pd.DataFrame({"npi": npi, "fold": f0})
assert chk.groupby("npi").fold.nunique().max() == 1, "NPI leaks across folds"
print("b4 fraud rows per fold:", pd.Series(is_b4).groupby(f0).sum().to_dict())
del chk

X_all: (7425062, 482) 14.3 GB


b4 fraud rows per fold: {0: 374, 1: 266, 2: 369, 3: 327, 4: 463}


## Step 2 — Train both models on the same folds, score both test sets

In [4]:
def scores(y, p, thr):
    if y.min() == y.max():
        return None
    pred = p >= thr
    tp = (pred & (y == 1)).sum(); fn = (~pred & (y == 1)).sum()
    tn = (~pred & (y == 0)).sum(); fp = (pred & (y == 0)).sum()
    tpr, tnr = tp / (tp + fn), tn / (tn + fp)
    return {"AUC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p), "base_rate": y.mean(),
            "TPR": tpr, "TNR": tnr, "GMean": np.sqrt(tpr * tnr), "test_pos": int(y.sum())}

done = set()
if RES_CSV.exists():
    prev = pd.read_csv(RES_CSV)
    done = set(zip(prev.run, prev.fold, prev.train))
print("Already done:", len(done))

TYPES = ["1128a1", "1128a2", "1128a3", "1128b4", "1128b7"]
for run in range(N_RUNS):
    folds = assign_folds(run)
    for k in FOLDS_TO_RUN:
        te = np.where(folds == k)[0]
        tr_all = np.where(folds != k)[0]
        y_te, g_te, t_te, b4_te = y_all[te], grp[te], pos_type[te], is_b4[te]
        for model_name in TRAIN_MODELS:
            if (run, k, model_name) in done:
                continue
            tr = tr_all if model_name == "train_full" else tr_all[~is_b4[tr_all]]
            thr = y_all[tr].mean()
            t0 = time.time()
            m = make_model(run).fit(X_all[tr], y_all[tr])
            p = m.predict_proba(X_all[te])[:, 1]
            sec = round(time.time() - t0, 1)
            rows = []
            for test_set, keep in [("test_full", np.ones(len(te), bool)), ("test_nob4", ~b4_te)]:
                for g in ["all", "both", "B_only", "D_only"]:
                    sel = keep if g == "all" else keep & (g_te == g)
                    r = scores(y_te[sel], p[sel], thr)
                    if r: rows.append({"run": run, "fold": k, "train": model_name, "test": test_set,
                                       "group": g, **r, "fit_sec": sec, "train_pos": int(y_all[tr].sum())})
            for T in TYPES:                                   # one exclusion type's positives vs all negatives
                sel = (y_te == 0) | (t_te == T)
                r = scores(y_te[sel], p[sel], thr)
                if r: rows.append({"run": run, "fold": k, "train": model_name, "test": f"type_{T}",
                                   "group": "all", **r, "fit_sec": sec, "train_pos": int(y_all[tr].sum())})
            pd.DataFrame(rows).to_csv(RES_CSV, mode="a", header=not RES_CSV.exists(), index=False)
            a = {(r["test"], r["group"]): r["AUC"] for r in rows}
            print(f"run{run} fold{k} {model_name}: full={a[('test_full','all')]:.4f} "
                  f"nob4={a[('test_nob4','all')]:.4f} b4-only={a.get(('type_1128b4','all'), np.nan):.4f} ({sec}s)")

Already done: 0


run0 fold0 train_full: full=0.7701 nob4=0.7985 b4-only=0.7377 (136.8s)


run0 fold0 train_nob4: full=0.7565 nob4=0.7984 b4-only=0.7087 (138.2s)


run0 fold1 train_full: full=0.7600 nob4=0.7826 b4-only=0.7151 (137.6s)


run0 fold1 train_nob4: full=0.7380 nob4=0.7715 b4-only=0.6714 (138.0s)


...

run5 fold2 train_nob4: full=0.7443 nob4=0.7940 b4-only=0.6926 (138.5s)


run5 fold3 train_full: full=0.7739 nob4=0.8104 b4-only=0.7276 (138.9s)


run5 fold3 train_nob4: full=0.7652 nob4=0.8165 b4-only=0.7003 (139.4s)


run5 fold4 train_full: full=0.7505 nob4=0.7704 b4-only=0.7255 (139.4s)


run5 fold4 train_nob4: full=0.7527 nob4=0.7974 b4-only=0.6965 (140.3s)


## Step 3 — Results

### 3a. The 2 × 2 table (all providers)

In [5]:
res = pd.read_csv(RES_CSV)
print("folds per model:", res[res.test == "test_full"].groupby("train")[["run", "fold"]].apply(lambda d: len(d.drop_duplicates())).to_dict())

def ci(s):
    m = s.mean(); h = 1.96 * s.std(ddof=1) / np.sqrt(len(s)) if len(s) > 1 else np.nan
    return f"{m:.4f} ({m-h:.4f}, {m+h:.4f})"

main = res[res.test.isin(["test_full", "test_nob4"])]
t = main[main.group == "all"].pivot_table(index="train", columns="test", values="AUC", aggfunc=ci)
print("AUC, all providers — rows: training labels, columns: test labels\n", t.to_string())
t.to_csv(RES_DIR / f"{TAG}_2x2_AUC.csv")

folds per model: {'train_full': 30, 'train_nob4': 30}
AUC, all providers — rows: training labels, columns: test labels
 test                      test_full                test_nob4
train                                                       
train_full  0.7634 (0.7583, 0.7685)  0.7996 (0.7937, 0.8054)
train_nob4  0.7529 (0.7483, 0.7576)  0.8033 (0.7980, 0.8086)


### 3b. Paired difference: does training without b4 help on the full test?

Same folds, same test rows, so the per-fold difference is a paired comparison.

In [6]:
for test_set in ["test_full", "test_nob4"]:
    for g in ["all", "both", "B_only", "D_only"]:
        w = main[(main.test == test_set) & (main.group == g)].pivot_table(index=["run", "fold"], columns="train", values="AUC")
        if w.empty or w.shape[1] < 2: continue
        d = (w["train_nob4"] - w["train_full"]).dropna()
        h = 1.96 * d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else np.nan
        print(f"{test_set:>9} | {g:>6}: nob4 − full = {d.mean():+.4f} (95% CI {d.mean()-h:+.4f}, {d.mean()+h:+.4f}); "
              f"better in {(d > 0).sum()}/{len(d)} folds")

test_full |    all: nob4 − full = -0.0105 (95% CI -0.0139, -0.0070); better in 5/30 folds
test_full |   both: nob4 − full = -0.0041 (95% CI -0.0098, +0.0016); better in 13/30 folds
test_full | B_only: nob4 − full = -0.0107 (95% CI -0.0194, -0.0020); better in 10/30 folds
test_full | D_only: nob4 − full = -0.0117 (95% CI -0.0172, -0.0062); better in 6/30 folds
test_nob4 |    all: nob4 − full = +0.0037 (95% CI -0.0011, +0.0085); better in 21/30 folds
test_nob4 |   both: nob4 − full = +0.0096 (95% CI +0.0021, +0.0170); better in 18/30 folds
test_nob4 | B_only: nob4 − full = -0.0010 (95% CI -0.0136, +0.0116); better in 14/30 folds
test_nob4 | D_only: nob4 − full = +0.0055 (95% CI -0.0013, +0.0122); better in 18/30 folds


### 3c. Per group

In [7]:
for metric in ["AUC", "AUPRC"]:
    t = main.pivot_table(index=["train", "test"], columns="group", values=metric, aggfunc="mean").round(4)
    print(f"\n{metric} (mean over folds)\n", t.reindex(columns=["all", "both", "B_only", "D_only"]).to_string())
    t.to_csv(RES_DIR / f"{TAG}_groups_{metric}.csv")


AUC (mean over folds)
 group                    all    both  B_only  D_only
train      test                                     
train_full test_full  0.7634  0.7691  0.7095  0.7647
           test_nob4  0.7996  0.8041  0.7776  0.7842
train_nob4 test_full  0.7529  0.7649  0.6988  0.7529
           test_nob4  0.8033  0.8137  0.7766  0.7897

AUPRC (mean over folds)
 group                    all    both  B_only  D_only
train      test                                     
train_full test_full  0.0038  0.0046  0.0040  0.0050
           test_nob4  0.0027  0.0034  0.0059  0.0030
train_nob4 test_full  0.0034  0.0038  0.0043  0.0051
           test_nob4  0.0029  0.0030  0.0065  0.0033


### 3d. How detectable is each exclusion type?

Each type's positives against all negatives, on the full test set. ~0.5 = indistinguishable from normal providers. `train_nob4` never saw b4 positives, so its b4 column is a pure test of whether license revocations look like the other fraud.

In [8]:
ty = res[res.test.str.startswith("type_")].assign(type=lambda d: d.test.str[5:])
t = ty.pivot_table(index="type", columns="train", values="AUC", aggfunc=ci)
n = ty[ty.train == "train_full"].groupby("type").test_pos.sum().rename("positives (all folds)")
print(t.join(n).to_string())
t.join(n).to_csv(RES_DIR / f"{TAG}_by_type_AUC.csv")

                     train_full               train_nob4  positives (all folds)
type                                                                           
1128a1  0.8116 (0.8048, 0.8183)  0.8171 (0.8105, 0.8238)                   9132
1128a2  0.7700 (0.7479, 0.7921)  0.7320 (0.7120, 0.7521)                   1032
1128a3  0.7749 (0.7576, 0.7923)  0.7788 (0.7634, 0.7942)                   2052
1128b4  0.7204 (0.7130, 0.7278)  0.6929 (0.6855, 0.7003)                  10794
1128b7  0.7693 (0.7378, 0.8007)  0.7972 (0.7571, 0.8374)                    738


## How to read the results

- **`train_nob4` beats `train_full` on `test_full`** (3b, CI above 0) → removing b4 from training gives a better model even by the paper's own label definition. Strongest claim.
- **About equal on `test_full`** → cleaner training costs nothing; the 0.804 mostly reflects the narrower test. Report it as a narrower task.
- **`train_full` better on `test_full`** → b4 examples carried useful signal; keep them in training.
- **3d:** if `1128b4` AUC is near 0.5 while a1 / a3 / b7 are clearly higher, license revocations are not detectable from billing — direct evidence for the label decision. Check `1128a2` (patient abuse) the same way.